# LiteRT Conversion

In [6]:
import os
# import onnx
import numpy as np 
import tensorflow as tf
import logging
import tflite
# from tqdm import tqdm
# from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score
from tensorflow.keras.preprocessing import image_dataset_from_directory
# from onnx_tf.backend import prepare


In [ ]:
def export_onnx_to_tf(onnx_path: str, tf_path: str) -> None:
    """
    Convert an ONNX model to TensorFlow format and export it as a SavedModel.

    This function loads an ONNX model from the specified path, converts it 
    to TensorFlow format using the ONNX-TF backend, and saves the resulting 
    TensorFlow model as a SavedModel to the given output path.

    Args:
        onnx_path (str): The file path to the ONNX model.
        tf_path (str): The file path where the converted TensorFlow model 
                       (SavedModel) will be saved.

    Returns:
        None: This function does not return any value. It exports the model to 
              the specified location.
    """
    # Load the ONNX model.
    onnx_model = onnx.load(onnx_path)
    
    # Convert to TensorFlow representation.
    tf_rep = prepare(onnx_model)
    
    # Export the model as a SavedModel.
    tf_rep.export_graph(tf_path)

def normalize_channels_first(x: tf.Tensor) -> tf.Tensor:
    """
    Normalizes a tensor in channels-first format using the ImageNet mean and std.
    
    Args:
        x (tf.Tensor): Input tensor of shape (batch, 3, height, width) with values in [0, 1].
    
    Returns:
        tf.Tensor: Normalized tensor.
    """
    # Define the mean and std as constants.
    mean = tf.constant([0.485, 0.456, 0.406], shape=(3, 1, 1), dtype=tf.float32)
    std = tf.constant([0.229, 0.224, 0.225], shape=(3, 1, 1), dtype=tf.float32)
    return (x - mean) / std

### Test Inference and Calculate Metrics

In [3]:
def calculate_metrics(y_true, y_pred, y_proba = None) -> dict[str, float]:
    """
    Calculates classification metrics, supporting both binary and multiclass classification.

    Parameters:
        y_true (list or ndarray): True labels.
        y_pred (list or ndarray): Predicted labels.
        y_proba (ndarray, optional): Predicted probabilities or scores for all classes 
                                     (required for AUC-Score computation).

    Returns:
        dict[str, float]: A dictionary containing the computed metrics.
    """
    # Determine whether the task is binary or multiclass.
    num_classes = len(set(y_true))
    average = 'binary' if num_classes == 2 else 'macro'

    metrics = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred, average=average),
        "Precision": precision_score(y_true, y_pred, average=average),
        "F1-Score": f1_score(y_true, y_pred, average=average),
    }

    if y_proba is not None:
        if num_classes == 2:
            # For binary, assume probabilities for the positive class are in column index 1.
            metrics["AUC-Score"] = roc_auc_score(y_true, y_proba[:, 0])
        else:
            # For multiclass, compute the AUC using one-vs-rest probabilities.
            metrics["AUC-Score"] = roc_auc_score(y_true, y_proba, multi_class='ovr', average='macro')

    return metrics

def test_inference_savedmodel(
    saved_model_path: str,
    data_dir: str,
    batch_size: int = 32,
    image_size: tuple = (224, 224),
    save_dir: str = None
) -> None:
    """
    Performs inference on the test dataset using a TensorFlow SavedModel,
    computes metrics, and optionally saves plots.

    The test dataset is loaded from a directory structure containing "train", "val", and "test" folders.
    It assumes that the test data is in the "test" folder within data_dir.

    Args:
        saved_model_path (str): Path to the TensorFlow SavedModel directory.
        data_dir (str): Root directory containing "train", "val", and "test" subdirectories.
        batch_size (int): Batch size for the test dataset. Defaults to 32.
        image_size (tuple): Desired image size (height, width). Defaults to (224, 224).
        save_dir (str): Directory to save plots. If None, plots will not be saved.
    """
    logging.info("Loading SavedModel for inference...")
    # Load the SavedModel; note that this does not return a Keras model with predict()
    model = tf.saved_model.load(saved_model_path)
    
    # Get the serving signature; this is a callable that accepts a dict of inputs.
    infer = model.signatures["serving_default"]
    
    # Determine the input key from the signature.
    input_key = list(infer.structured_input_signature[1].keys())[0]
    logging.info(f"Model input key: {input_key}")
    
    # Load the test dataset from the "test" folder.
    test_dir = os.path.join(data_dir, "test")
    test_dataset = image_dataset_from_directory(
        test_dir,
        labels="inferred",
        label_mode="int",
        batch_size=batch_size,
        image_size=image_size,
        shuffle=False  # For reproducible order.
    )
    class_labels = test_dataset.class_names
    logging.info(f"Found classes: {class_labels}")
    
    # Preprocessing: 
    # 1. Rescale images from [0, 255] to [0, 1].
    normalization_layer = tf.keras.layers.Rescaling(1./255)
    # 2. Transpose images from channels-last (H, W, C) to channels-first (C, H, W).
    # 3. Normalize using the same mean and std as during training.
    test_dataset = test_dataset.map(lambda x, y: (
        normalize_channels_first(tf.transpose(normalization_layer(x), perm=[0, 3, 1, 2])),
        y
    ))
    
    predictions_list = []
    ground_truths_list = []
    
    logging.info("Running inference on the test dataset using the serving signature...")
    # Iterate over the dataset and call the signature on each batch.
    for inputs, labels in tqdm(test_dataset, desc="TF Inference Progress"):
    # for inputs, labels in test_dataset:
        # Call the model's serving signature with the proper input key.
        outputs = infer(**{input_key: inputs})
        # Assume the first output is the one we need.
        output_key = list(outputs.keys())[0]
        batch_predictions = outputs[output_key]
        predictions_list.append(batch_predictions.numpy())
        ground_truths_list.append(labels.numpy())
    
    # Concatenate all predictions and ground truths.
    predictions = np.concatenate(predictions_list, axis=0)
    ground_truths = np.concatenate(ground_truths_list, axis=0)

    # Process predictions depending on the number of classes.
    if len(class_labels) == 2:
        # For binary classification: apply sigmoid and threshold at 0.5.
        probs = tf.nn.sigmoid(predictions).numpy()
        pred_labels = (probs >= 0.5).astype(int).flatten()
    else:
        # For multi-class classification: apply softmax and use argmax.
        probs = tf.nn.softmax(predictions, axis=-1).numpy()
        pred_labels = np.argmax(probs, axis=-1)
    
    # Calculate metrics using your helper function.
    metrics: dict[str, float] = calculate_metrics(ground_truths, pred_labels, probs)
    
    # Print the metrics.
    print("Metrics Results:")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")

    # Generate and save plots if a save directory is provided.
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        plot_metric_bar(metrics, save_path=os.path.join(save_dir, "metrics_bar_chart.png"))
        if class_labels:
            plot_confusion_matrix(
                y_true=ground_truths,
                y_pred=pred_labels,
                labels=class_labels,
                save_path=os.path.join(save_dir, "confusion_matrix.png")
            )
        if probs is not None:
            plot_roc_auc_curve(ground_truths, probs, save_path=os.path.join(save_dir, "roc_auc_curve.png"))
        plot_radar_chart(metrics, save_path=os.path.join(save_dir, "radar_chart.png"))
        logging.info(f"Plots saved to {save_dir}")

In [4]:
saved_model_dir = "model_tf"  # Path to your SavedModel directory.
data_directory = "data/skin-lesions/download"
loaded_model = tf.saved_model.load(saved_model_dir)
infer = loaded_model.signatures["serving_default"]
print(infer.structured_input_signature)

((), {'input': TensorSpec(shape=(None, 3, 224, 224), dtype=tf.float32, name='input')})


In [5]:
test_inference_savedmodel(saved_model_dir, data_directory, batch_size=32, image_size=(224, 224))

Found 3674 files belonging to 14 classes.




TF Inference Progress: 100%|██████████| 115/115 [03:38<00:00,  1.90s/it]

Metrics Results:
Accuracy: 0.7327
Recall: 0.5684
Precision: 0.6812
F1-Score: 0.5989
AUC-Score: 0.9623


### Convert SavedModel to LiteRT

In [13]:
def convert_savedmodel_to_tflite(saved_model_dir: str, tflite_model_path: str) -> None:
    """
    Convert a TensorFlow SavedModel to TensorFlow Lite format.

    Args:
        saved_model_dir (str): Path to the TensorFlow SavedModel directory.
        tflite_model_path (str): Path to save the converted TensorFlow Lite model.
    """
    # Load the SavedModel and create a TFLiteConverter.
    converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
    
    # Enable the new converter (if not already default) and allow select TensorFlow ops.
    converter.experimental_new_converter = True
    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS,   # Use built-in TFLite ops.
        # tf.lite.OpsSet.SELECT_TF_OPS      # Fallback for ops not natively supported.
    ]
    
    # Optionally, you can enable optimizations (e.g., full integer quantization)
    # converter.optimizations = [tf.lite.Optimize.DEFAULT]
    
    try:
        tflite_model = converter.convert()
        with open(tflite_model_path, "wb") as f:
            f.write(tflite_model)
        print(f"Model converted and saved to {tflite_model_path}")
    except Exception as e:
        print(f"Error converting the model: {e}")

In [14]:
saved_model_dir = "model_tf"  # Path to your SavedModel directory.
tflite_model_path = "model.tflite"
convert_savedmodel_to_tflite(saved_model_dir=saved_model_dir, tflite_model_path=tflite_model_path)

Error converting the model: Could not translate MLIR to FlatBuffer.<unknown>:0: error: loc(callsite(callsite(fused["ClipByValue:", "saturate_cast/clamp@__inference___call___2813"] at fused["PartitionedCall:", "PartitionedCall@__inference_signature_wrapper_2829"]) at fused["PartitionedCall:", "PartitionedCall"])): 'tf.ClipByValue' op is neither a custom op nor a flex op
<unknown>:0: note: loc(fused["PartitionedCall:", "PartitionedCall"]): called from
<unknown>:0: note: loc(callsite(callsite(fused["ClipByValue:", "saturate_cast/clamp@__inference___call___2813"] at fused["PartitionedCall:", "PartitionedCall@__inference_signature_wrapper_2829"]) at fused["PartitionedCall:", "PartitionedCall"])): see current operation: %252 = "tf.ClipByValue"(%251, %178, %180) {device = ""} : (tensor<?x3x224x224xf32>, tensor<f32>, tensor<f32>) -> tensor<?x3x224x224xf32>
<unknown>:0: note: loc(callsite(callsite(fused["ClipByValue:", "saturate_cast/clamp@__inference___call___2813"] at fused["PartitionedCall:"

W0000 00:00:1743284276.621180   16977 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1743284276.621201   16977 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
2025-03-29 17:37:56.621333: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: model_tf
2025-03-29 17:37:56.624716: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2025-03-29 17:37:56.624724: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: model_tf
2025-03-29 17:37:56.650815: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2025-03-29 17:37:56.694143: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: model_tf
2025-03-29 17:37:56.737103: I tensorflow/cc/saved_model/loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 115775 microseconds.
loc(callsite(callsite(fused["ClipByValue:", "saturate_cast/clamp@__i

In [12]:

def get_builtin_operator_map() -> dict[int, str]:
    """
    Builds a dictionary mapping numeric operator codes to their string names
    by inspecting tflite.BuiltinOperator.
    
    Returns:
        dict[int, str]: Mapping from op code to op name.
    """
    op_map: dict[int, str] = {}
    for attr_name in dir(tflite.BuiltinOperator):
        if not attr_name.startswith("__"):
            attr_value = getattr(tflite.BuiltinOperator, attr_name)
            if isinstance(attr_value, int):
                op_map[attr_value] = attr_name
    return op_map

def report_tflite_ops(tflite_model_path: str) -> None:
    """
    Reads a TFLite model file and prints a report of the operators in the model.
    It reports the total number of operators, counts for TFLite built-in ops,
    and counts for fallback (SELECT_TF_OPS) ops.
    
    If an operator's built-in code is CUSTOM, this function checks its custom code.
    If the custom code begins with 'Flex', the operator is considered a fallback op.
    
    Args:
        tflite_model_path (str): Path to the .tflite model file.
    """
    if not os.path.exists(tflite_model_path):
        logging.error(f"File {tflite_model_path} not found.")
        return

    with open(tflite_model_path, "rb") as f:
        buf = f.read()

    # Load the FlatBuffer TFLite model.
    model = tflite.Model.GetRootAsModel(buf, 0)
    total_ops = 0
    builtin_ops = 0
    fallback_ops = 0  # Counting SELECT_TF_OPS/fallback ops.
    op_counts: dict[str, int] = {}
    select_tf_ops: set[str] = set()

    # Build our mapping from op code to name for built-in ops.
    op_map = get_builtin_operator_map()

    # Iterate over each subgraph in the model.
    # A TFLite model can contain multiple subgraphs, each representing a separate computational graph.
    for subgraph_idx in range(model.SubgraphsLength()):
        subgraph = model.Subgraphs(subgraph_idx)           # Retrieve the subgraph at index `subgraph_idx`.
        for op_idx in range(subgraph.OperatorsLength()):   # Iterate over all the operators (layers/operations) in the current subgraph.
            op = subgraph.Operators(op_idx)                # Retrieve the operator at index `op_idx`.
            opcode_index = op.OpcodeIndex()                # Get the opcode index, which is an identifier referring to the type of operation.
            op_code = model.OperatorCodes(opcode_index)    # Retrieve the operator code information associated with this operation.
            builtin_code = op_code.BuiltinCode()           # Extract the built-in operation code (an integer that maps to a specific operation type).

            # Check if the operator is a custom op. If so, inspect its custom code.
            if builtin_code == tflite.BuiltinOperator.CUSTOM:
                custom_code = op_code.CustomCode()          # Retrieve the custom operation's string identifier (if available).
                # If the custom code indicates a fallback TensorFlow op, count it as SELECT_TF_OPS.
                # If the custom code starts with "Flex", it indicates a fallback to a TensorFlow op.
                if custom_code is not None and custom_code.startswith(b"Flex"): 
                    op_name = "SELECT_TF_OPS"               # Mark it as a TensorFlow fallback operation.
                    fallback_ops += 1
                    select_tf_ops.add(custom_code)
                else:
                    op_name = "CUSTOM"                      # Otherwise, classify it as a general custom operation.
                    builtin_ops += 1                        # Count it as a built-in op (or modify if treating custom separately).
            else:
                # For non-custom (standard TFLite) operations, look up the operation name using `op_map`.
                op_name = op_map.get(builtin_code, "UNKNOWN")
                builtin_ops += 1

            # Update the operator count dictionary to track occurrences of each type of operation.
            op_counts[op_name] = op_counts.get(op_name, 0) + 1
            total_ops += 1

    print("TFLite Operator Conversion Report:")
    print(f"Total operators: {total_ops}")
    print(f"TFLite Built-in ops: {builtin_ops}")
    print(f"SELECT_TF_OPS (fallback ops): {fallback_ops}")
    print(f"SELECT_TF_OPS (fallback ops): {select_tf_ops}")
    print("\nDetailed operator counts:")
    for op_name, count in op_counts.items():
        print(f"{op_name}: {count}")

# Example usage
saved_model_dir = "model.tflite"
report_tflite_ops(saved_model_dir)

TFLite Operator Conversion Report:
Total operators: 648
TFLite Built-in ops: 582
SELECT_TF_OPS (fallback ops): 66
SELECT_TF_OPS (fallback ops): {b'FlexClipByValue'}

Detailed operator counts:
MUL: 131
ROUND: 66
ADD: 39
SELECT_TF_OPS: 66
CAST: 132
SUB: 29
PAD: 18
TRANSPOSE: 104
CONV_2D: 35
DEPTHWISE_CONV_2D: 17
MEAN: 1
SHAPE: 2
STRIDED_SLICE: 2
REDUCE_PROD: 1
PACK: 2
RESHAPE: 2
FULLY_CONNECTED: 1


In [ ]:
print(os.path.getsize("quantized_student.onnx") / 1e6)
print(os.path.getsize("model.tflite") / 1e6)
print(os.path.getsize("models/SkinCancer/Quantized/quantized_student_state.pth") / 1e6)


2.392615
9.027004


In [20]:
def get_model_size(model_dir):
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(model_dir):
        for filename in filenames:
            filepath = os.path.join(dirpath, filename)
            total_size += os.path.getsize(filepath)
    return total_size

# Path to your SavedModel directory
model_path = 'model_tf'

size_in_bytes = get_model_size(model_path)
print(f"Model size: {size_in_bytes / (1024**2):.2f} MB")

Model size: 4.96 MB


In [ ]:
import torch

def inspect_pth_weights(pth_file: str) -> None:
    """
    Loads a PyTorch model from a .pth file and checks how many tensors are quantized (INT8) vs. non-quantized (FP32).

    Args:
        pth_file (str): Path to the .pth file containing the model weights.
    """
    # Load the model checkpoint
    checkpoint = torch.load(pth_file, map_location="cpu")

    # Initialize counters
    quantized_count = 0
    non_quantized_count = 0
    total_tensors = 0

    for name, tensor in checkpoint.items():
        if isinstance(tensor, torch.Tensor):
            total_tensors += 1  # Count total tensors
            if tensor.is_quantized:
                quantized_count += 1
                print(f"Quantized Tensor: {name}, dtype: {tensor.dtype}, shape: {tensor.shape}, qscheme: {tensor.qscheme()}")
            else:
                non_quantized_count += 1
                print(f"Floating-point Tensor: {name}, dtype: {tensor.dtype}, shape: {tensor.shape}")

    # Print summary
    print("\n===== Summary =====")
    print(f"Total Tensors: {total_tensors}")
    print(f"Quantized (INT8) Tensors: {quantized_count}")
    print(f"Non-Quantized (FP32) Tensors: {non_quantized_count}")
    
    if quantized_count > 0:
        print("\nThe model contains quantized weights.")
    else:
        print("\nNo quantized tensors detected. The model might have been converted back to FP32.")


# Example usage:
inspect_pth_weights("models/SkinCancer/Quantized/quantized_student_state.pth")

Floating-point Tensor: features_0_0_input_scale_0, dtype: torch.float32, shape: torch.Size([])
Floating-point Tensor: features_0_0_input_zero_point_0, dtype: torch.int64, shape: torch.Size([])
Floating-point Tensor: features_3_scale_0, dtype: torch.float32, shape: torch.Size([])
Floating-point Tensor: features_3_zero_point_0, dtype: torch.int64, shape: torch.Size([])
Floating-point Tensor: features_5_scale_0, dtype: torch.float32, shape: torch.Size([])
Floating-point Tensor: features_5_zero_point_0, dtype: torch.int64, shape: torch.Size([])
Floating-point Tensor: features_6_scale_0, dtype: torch.float32, shape: torch.Size([])
Floating-point Tensor: features_6_zero_point_0, dtype: torch.int64, shape: torch.Size([])
Floating-point Tensor: features_8_scale_0, dtype: torch.float32, shape: torch.Size([])
Floating-point Tensor: features_8_zero_point_0, dtype: torch.int64, shape: torch.Size([])
Floating-point Tensor: features_9_scale_0, dtype: torch.float32, shape: torch.Size([])
Floating-poi

In [ ]:
#TODO: ops mightve turned back to fp32, must convert to int8 with post-training quanization
#TODO: check ai_edge_torch
# model = tf.saved_model.load(model_path)

# # Check the data type of the model's weights
# for variable in model.trainable_variables:
#     print(variable.name, variable.dtype)

AttributeError: '_UserObject' object has no attribute 'trainable_variables'